In [1]:
# Export the trained U-Net (best.pth) to ONNX + collect all final model files in one place
!pip install -q segmentation-models-pytorch onnx onnxscript onnxruntime

import glob, os, shutil, numpy as np, torch, onnxruntime as ort
import segmentation_models_pytorch as smp

OUT = "/kaggle/working"
pth = glob.glob("/kaggle/input/**/best.pth", recursive=True)
assert pth, "best.pth not found - Add Input > Notebooks > flood_unet_training.ipynb"
src = os.path.dirname(pth[0])
print("Using:", pth[0])

# same architecture as training; weights come from best.pth
model = smp.Unet("resnet34", encoder_weights=None, in_channels=6, classes=3)
model.load_state_dict(torch.load(pth[0], map_location="cpu"))
model.eval()

dummy = torch.randn(1, 6, 256, 256)
torch.onnx.export(model, dummy, f"{OUT}/flood_unet.onnx",
                  input_names=["x"], output_names=["logits"],
                  dynamic_axes={"x": {0: "b", 2: "h", 3: "w"}, "logits": {0: "b", 2: "h", 3: "w"}},
                  opset_version=17, dynamo=False)

# verify ONNX gives the same answer as PyTorch
sess = ort.InferenceSession(f"{OUT}/flood_unet.onnx")
with torch.no_grad():
    ref = model(dummy).numpy()
diff = float(np.abs(sess.run(None, {"x": dummy.numpy()})[0] - ref).max())
print("ONNX export OK, max difference vs PyTorch:", diff)

# copy the other training outputs next to the ONNX file
for name in ("best.pth", "norm.json", "metrics.json", "training_curves.png"):
    p = os.path.join(src, name)
    if os.path.exists(p):
        shutil.copy(p, OUT)
print("Final files:", sorted(f for f in os.listdir(OUT) if not f.startswith("__")))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 4.3 MB/s eta 0:00:00
Using: /kaggle/input/notebooks/gayatri226/flood-unet-training-ipynb/best.pth


/tmp/ipykernel_16/410114285.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, f"{OUT}/flood_unet.onnx",


ONNX export OK, max difference vs PyTorch: 4.5299530029296875e-06
Final files: ['best.pth', 'flood_unet.onnx', 'metrics.json', 'norm.json', 'training_curves.png']


In [2]:
!ls -R /kaggle/input | head -40

/kaggle/input:
notebooks

/kaggle/input/notebooks:
gayatri226

/kaggle/input/notebooks/gayatri226:
flood-unet-training-ipynb

/kaggle/input/notebooks/gayatri226/flood-unet-training-ipynb:
best.pth
custom.css
__huggingface_repos__.json
metrics.json
norm.json
__notebook__.ipynb
__output__.json
__results___files
__results__.html
training_curves.png

/kaggle/input/notebooks/gayatri226/flood-unet-training-ipynb/__results___files:
__results___9_0.png
